# 02 - Requetes Cypher de Base

Ce notebook introduit les requetes Cypher fondamentales pour interroger un graphe Neo4j.

## Concepts couverts
- MATCH et RETURN
- WHERE (filtrage)
- Relations et traversees
- Agregations

In [2]:
! pip install neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 180.1 kB/s eta 0:00:0000:0100:01


In [3]:
import os
from neo4j import GraphDatabase
import pandas as pd

NEO4J_URI = os.environ.get('NEO4J_URI', 'bolt://neo4j:7687')
NEO4J_USER = os.environ.get('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.environ['NEO4J_PASSWORD']

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def query(cypher, params=None):
    with driver.session() as session:
        result = session.run(cypher, params or {})
        return pd.DataFrame([dict(r) for r in result])

## 1. MATCH simple - Trouver des noeuds

In [4]:
# Trouver tous les variants COVID
query("""
MATCH (d:Disease)
RETURN d.variant AS variant, d.severity AS severite, d.transmission_rate AS taux
ORDER BY d.emergence_date
""")

,variant,severite,taux
0,Original (Wuhan),moderate,0.25
1,Alpha (B.1.1.7),moderate,0.40
2,Delta (B.1.617.2),severe,0.60
3,Omicron (BA.1),mild,0.85
4,Omicron (BA.5),mild,0.90


## 2. WHERE - Filtrer les resultats

In [5]:
# Patients ages de plus de 65 ans non vaccines
query("""
MATCH (p:Patient)
WHERE p.age > 65 AND p.vaccination_status = 'unvaccinated'
RETURN p.patient_id, p.age, p.comorbidities
LIMIT 10
""")

,p.patient_id,p.age,p.comorbidities
0,P000145,66,none
1,P000230,66,cardiac
2,P000241,66,respiratory
3,P000412,66,diabetes
4,P000467,66,cardiac
5,P000606,66,diabetes
6,P000673,66,diabetes
7,P000823,66,none
8,P000892,66,none
9,P000989,66,hypertension


## 3. Relations - Traverser le graphe

In [6]:
# Patients et leur localisation
query("""
MATCH (p:Patient)-[:LIVES_IN]->(l:Location)
RETURN p.patient_id, p.age, l.city, l.country
LIMIT 10
""")

,p.patient_id,p.age,l.city,l.country
0,P000381,52,French Guiana,France
1,H001924,54,French Guiana,France
2,H009920,70,French Guiana,France
3,H003221,21,French Guiana,France
4,H001613,23,French Guiana,France
5,H003774,84,French Guiana,France
6,H002586,86,French Guiana,France
7,H004780,40,French Guiana,France
8,H002342,52,French Guiana,France
9,H009467,37,French Guiana,France


In [7]:
# Chaine d'infection directe
query("""
MATCH (infecteur:Patient)-[r:INFECTED_BY]->(infecte:Patient)
RETURN infecteur.patient_id AS source, 
       infecte.patient_id AS cible,
       r.date AS date_infection
LIMIT 15
""")

,source,cible,date_infection
0,P000025,P000291,2020-03-03
1,P000064,P000068,2020-03-09
2,P000068,P000109,2020-03-08
3,P000078,P000109,2020-03-08
4,P000088,P000435,2020-03-11
5,P000109,P000458,2020-03-07
6,P000166,P000397,2020-03-08
7,P000200,P000456,2020-03-05
8,P000230,P000000,2020-03-01
9,P000291,P000230,2020-03-02


## 4. Chemins de longueur variable

In [10]:
# Chaine de contamination sur 3 niveaux
query("""
MATCH path = (p:Patient {patient_id: 'P000000'})-[:INFECTED_BY*1..3]->(contact)
RETURN contact.patient_id AS contact, length(path) AS profondeur
LIMIT 20
""")

""


## 5. Agregations

In [9]:
# Nombre d'infections par variant
query("""
MATCH (p:Patient)-[:HAS_DISEASE]->(d:Disease)
RETURN d.variant AS variant, count(p) AS cas, avg(p.age) AS age_moyen
ORDER BY cas DESC
""")

,variant,cas,age_moyen
0,Delta (B.1.617.2),10010,40.687712
1,Omicron (BA.1),8800,40.240682
2,Alpha (B.1.1.7),8020,40.702244
3,Omicron (BA.5),7200,40.596250
4,Original (Wuhan),6015,40.401995


In [ ]:
driver.close()